# PPO Signal Strategy - Research Notebook

Defines PPO (Percentage Price Oscillator) indicator math and signal logic for parity with Pine Script and engine strategies.

## Indicator Math

### PPO Formula
```
PPO = (ma_fast - ma_slow) / ma_slow * 100
```

### MA Types (matype)
- 0: SMA, 1: EMA, 2: WMA, 3: DEMA

### DEMA Formula
```
DEMA = 2 * EMA(close, period) - EMA(EMA(close, period), period)
```

### EMA Initialization
- **TA-Lib**: SMA of first `period` values as seed
- **Pine ta.ema**: First source value as seed
- **Engine (target)**: SMA of first `period` values

In [ ]:
import numpy as np
import pandas as pd
import talib
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load sample data (ETH-USD 1h)
ticker = yf.Ticker("ETH-USD")
df = ticker.history(interval="1h", start="2025-01-01", end="2025-02-15")
df = df.reset_index()
df.columns = [c.replace(" ", "_") for c in df.columns]
closes = df["Close"].values.astype(np.float64)
print(f"Loaded {len(df)} bars")

In [ ]:
# PPO with TA-Lib (fast=38, slow=205, DEMA=matype 3)
fast, slow, matype = 38, 205, 3
ppo_talib = talib.PPO(closes, fastperiod=fast, slowperiod=slow, matype=matype)
df["PPO_TA-Lib"] = ppo_talib

In [ ]:
def ppo_handrolled(closes, fast_period, slow_period, ma_type=3):
    """Hand-rolled PPO with SMA-seed EMA (matches TA-Lib convention)."""
    def ema_sma_seed(src, period):
        alpha = 2.0 / (period + 1)
        out = np.full_like(src, np.nan)
        buf = []
        ema_val = np.nan
        for i in range(len(src)):
            x = src[i]
            if np.isnan(x):
                continue
            if len(buf) < period:
                buf.append(x)
                if len(buf) == period:
                    ema_val = sum(buf) / period
                    out[i] = ema_val
            else:
                ema_val = (x - ema_val) * alpha + ema_val
                out[i] = ema_val
        return out
    
    if ma_type == 3:  # DEMA
        ema1 = ema_sma_seed(closes, fast_period)
        ema2 = ema_sma_seed(ema1, fast_period)
        ma_fast = 2 * ema1 - ema2
        ema1_s = ema_sma_seed(closes, slow_period)
        ema2_s = ema_sma_seed(ema1_s, slow_period)
        ma_slow = 2 * ema1_s - ema2_s
    else:
        raise NotImplementedError("Only matype=3 DEMA")
    
    ppo = np.where(ma_slow != 0, (ma_fast - ma_slow) / ma_slow * 100, np.nan)
    return ppo

In [ ]:
# Compare hand-rolled vs TA-Lib
ppo_hand = ppo_handrolled(closes, fast, slow, matype)
df["PPO_Hand"] = ppo_hand
valid = ~np.isnan(ppo_talib) & ~np.isnan(ppo_hand)
diff = np.abs(ppo_talib[valid] - ppo_hand[valid])
print(f"Max diff: {diff.max():.6f}")
print(f"Mean diff: {diff.mean():.6f}")
df[["Close", "PPO_TA-Lib", "PPO_Hand"]].tail(20)

## Signal Logic (momentum mode)

- **Long entry**: `ppo[1] < ppo_upper` and `ppo >= ppo_upper`
- **Short entry**: `ppo[1] > ppo_lower` and `ppo <= ppo_lower`
- **Exit**: flip on opposite signal, midpoint, max hold, or stop loss